# Customer Churn Prediction and Business Insights
End-to-end analysis and modeling workflow for Telco churn prediction.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_style('whitegrid')
RANDOM_STATE = 42

In [ ]:
data_path = Path('../data/churn.csv')

df = pd.read_csv(data_path)
print('Data path:', data_path)
display(df.head())
print('Shape:', df.shape)
df.info()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].mode()[0])

df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})
print(df.isna().sum().sort_values(ascending=False).head())

In [ ]:
Path('../visuals').mkdir(exist_ok=True)

plt.figure(figsize=(6, 4))
sns.countplot(x='Churn', data=df)
plt.title('Churn Distribution (0 = No, 1 = Yes)')
plt.tight_layout()
plt.savefig('../visuals/churn_distribution.png', dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title('MonthlyCharges vs Churn')
plt.tight_layout()
plt.savefig('../visuals/monthly_charges_vs_churn.png', dpi=300)
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x='Churn', y='tenure', data=df)
plt.title('Tenure vs Churn')
plt.tight_layout()
plt.savefig('../visuals/tenure_vs_churn.png', dpi=300)
plt.show()

corr_df = pd.get_dummies(df, drop_first=True)
plt.figure(figsize=(14, 10))
sns.heatmap(corr_df.corr(), cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('../visuals/correlation_heatmap.png', dpi=300)
plt.show()

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

log_pre = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
])

rf_pre = ColumnTransformer([
    ('num', 'passthrough', num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [ ]:
log_model = Pipeline([
    ('preprocessor', log_pre),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

rf_model = Pipeline([
    ('preprocessor', rf_pre),
    ('classifier', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE))
])

log_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f'\n{name} Accuracy: {acc:.4f}')
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'{name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'../visuals/{name.lower().replace(
, 
)}_confusion_matrix.png', dpi=300)
    plt.show()
    return acc

log_acc = evaluate_model('Logistic Regression', log_model, X_test, y_test)
rf_acc = evaluate_model('Random Forest', rf_model, X_test, y_test)

pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [log_acc, rf_acc]
}).sort_values('Accuracy', ascending=False)

In [ ]:
pre = rf_model.named_steps['preprocessor']
clf = rf_model.named_steps['classifier']

fi = pd.DataFrame({
    'feature': pre.get_feature_names_out(),
    'importance': clf.feature_importances_
}).sort_values('importance', ascending=False)

display(fi.head(12))

plt.figure(figsize=(10, 6))
sns.barplot(data=fi.head(12), x='importance', y='feature', palette='viridis')
plt.title('Top 12 Factors Influencing Churn')
plt.tight_layout()
plt.savefig('../visuals/top_feature_importance.png', dpi=300)
plt.show()

In [ ]:
best_model = rf_model if rf_acc >= log_acc else log_model
Path('../models').mkdir(exist_ok=True)
joblib.dump(best_model, '../models/best_churn_model.joblib')
print('Model saved at ../models/best_churn_model.joblib')